In [ ]:
import os

def get_last_row_as_list(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
        last_line = lines[-1].strip().split('\t')
    return last_line

def get_first_row_as_list(file_path):
    with open(file_path, 'r') as file:
        first_line = file.readline().strip().split('\t')
    return first_line

def get_N_row_as_list(file_path, N):
    with open(file_path, 'r') as file:
        lines = file.readlines()
        line = lines[N].strip().split('\t')
    return line

def get_most_recent_file(directory, prefix, extension):
    # Ottieni una lista di file nella directory
    files = os.listdir(directory)
    
    # Filtra i file che iniziano con il prefisso specificato e hanno l'estensione determinata
    files = [f for f in files if f.startswith(prefix) and f.endswith(extension) and os.path.isfile(os.path.join(directory, f))]
    
    if not files:
        return None
    
    # Ottieni il percorso completo del file più recente
    most_recent_file = max(files, key=lambda f: os.path.getmtime(os.path.join(directory, f)))
    
    return os.path.join(directory, most_recent_file)


In [ ]:
import pandas as pd
M = get_most_recent_file('/media/Mercury_dati/', 'Mercury_', '.txt')
df = pd.read_csv(most_recent_file, delimiter='\t', header=0)
df.tail(1)

In [ ]:
M = get_most_recent_file('/media/Mercury_dati/', 'Mercury_', '.txt')
header_M = get_first_row_as_list(M)
data = get_last_row_as_list(M)
print(header_M, data)

In [ ]:
header_M[1:]

In [ ]:
FP = get_most_recent_file('/media/avs-47/', 'logFP___', '.dat')
header_FP = get_first_row_as_list(FP)
data = get_last_row_as_list(FP)
print(header_FP, data)

In [ ]:
datas = get_last_row_as_list(FP)
data = [float(item) for item in datas[1:]]
print(data)

In [ ]:
header_FP[0:]

In [ ]:
AVS = get_most_recent_file('/media/avs-47/', 'LogAVS___', '.dat')
header_AVS = get_N_row_as_list(AVS, 2)
data = get_last_row_as_list(AVS)
print(header_AVS, data)

In [ ]:
import midas.client
import midas.frontend

class MyFrontend(midas.frontend.FrontendBase):
    def __init__(self):
        super().__init__("my_frontend")
        self.equipment = {
            "my_equipment": {
                "period": 1000,
                "readout": self.readout,
            }
        }

    def readout(self):
        # Crea un buffer di eventi
        event = self.create_event()
        
        # Scrivi i dati nel buffer
        data = {
            "temperature": 25.0,
            "pressure": 1013.25
        }
        event.add_bank("DATA", data)
        
        # Invia l'evento al sistema MIDAS
        self.send_event(event)

if __name__ == "__main__":
    frontend = MyFrontend()
    frontend.run()

In [ ]:
import midas
import midas.frontend
import midas.event
import numpy as np
equip_name = "SCFrontend"
M = get_most_recent_file('/media/Mercury_dati/', 'Mercury_', '.txt')
header_M = get_first_row_as_list(M)
client = midas.client.MidasClient("mytest")
buffer_handle = client.open_event_buffer("SYSTEM",None,1000000000)
request_id = client.register_event_request(buffer_handle, sampling_type = 2)
client.odb_set("/Equipment/{:}/Settings/Names {:}".format(equip_name, "MERC"), header_M[1:])
client.deregister_event_request(buffer_handle, request_id)
client.disconnect()

In [ ]:
"/Equipment/{:}/Settings/Names {:}".format(equip_name, "MERC")

In [ ]:
np.array([[1,4], [7,12]])

In [ ]:
a= [1,4]
b= [2,5]
np.concatenate((a,b))

In [ ]:
"""
A simple client that registers to receive events from midas.
"""

import midas.client
import numpy as np
import matplotlib.pyplot as plt

# Create our client
client = midas.client.MidasClient("notebook")

# Define which buffer we want to listen for events on (SYSTEM is the 
# main midas buffer).
buffer_handle = client.open_event_buffer("SYSTEM")

# Request events from this buffer that match certain criteria. In this
# case we will only be told about events with an "event ID" of 14.
request_id = client.register_event_request(buffer_handle, event_id = 4)
n = 0
while True:
    # If there's an event ready, `event` will contain a `midas.event.Event`
    # object. If not, it will be None. If you want to block waiting for an
    # event to arrive, you could set async_flag to False.
    event = client.receive_event(buffer_handle, async_flag=True)

    if event is not None:
        # Print some information to screen about this event.
        bank_names = ", ".join(b.name for b in event.banks.values())
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        event_number  = event.header.serial_number
        if 'SSIM' in bank_names:
            stream_len = len(event.banks['SSIM'].data) // 2
            t = np.array(event.banks['SSIM'].data)[:stream_len]
            x = np.array(event.banks['SSIM'].data)[stream_len:]
            print (stream_len, t[0:3], x[0:3], len(t), len(x))
            
            # Talk to midas so it knows we're alive, or can kill us if the user
            # pressed the "stop program" button.
            # Plot the displacement x(t)
            plt.figure(figsize=(8, 3))
            plt.subplot(1, 2, 1)
            plt.plot(t, x)
            plt.xlabel('Time (s)')
            plt.ylabel('Displacement x(t)')
            plt.title('Displacement')
            plt.grid(True)
            
            # Compute and plot the Fourier Transform of x(t)
            X_f = np.fft.fft(x)
            frequencies = np.fft.fftfreq(len(x), t[2]-t[1])
            
            plt.subplot(1, 2, 2)
            plt.plot(frequencies[:len(frequencies)//2], np.abs(X_f)[:len(X_f)//2])
            plt.xlabel('Frequency (Hz)')
            plt.ylabel('Magnitude')
            plt.title('Fourier Transform ')
            plt.grid(True)
            
            plt.tight_layout()
            plt.show()
            n +=1
        if n == 10:
            break
    client.communicate(10)
client.deregister_event_request(buffer_handle, request_id)
client.disconnect()
del client, buffer_handle, request_id

In [ ]:
"""
A simple client that registers to receive events from midas and testing wavelet
"""

import midas.client
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.fftpack import fft, fftfreq

# Create our client
client = midas.client.MidasClient("notebook")

# Define which buffer we want to listen for events on (SYSTEM is the 
# main midas buffer).
buffer_handle = client.open_event_buffer("SYSTEM")

# Request events from this buffer that match certain criteria. In this
# case we will only be told about events with an "event ID" of 14.
request_id = client.register_event_request(buffer_handle, event_id = 1)
n = 0
while True:
    # If there's an event ready, `event` will contain a `midas.event.Event`
    # object. If not, it will be None. If you want to block waiting for an
    # event to arrive, you could set async_flag to False.
    event = client.receive_event(buffer_handle, async_flag=True)

    if event is not None:
        # Print some information to screen about this event.
        bank_names = ", ".join(b.name for b in event.banks.values())
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        event_number  = event.header.serial_number
        if 'SSIM' in bank_names:
            stream_len = len(event.banks['SSIM'].data) // 2
            t = np.array(event.banks['SSIM'].data)[:stream_len]
            signal = np.array(event.banks['SSIM'].data)[stream_len:]
            
            ### 📌 TRASFORMATA DI FOURIER (FFT)
            fft_values = fft(signal)  # Trasformata di Fourier
            frequencies = fftfreq(len(t), t[2]-t[1])  # Assi delle frequenze
            
            # Consideriamo solo le frequenze positive
            positive_frequencies = frequencies[:len(frequencies)//2]
            fft_magnitudes = np.abs(fft_values[:len(frequencies)//2])
            
            ### 📌 TRASFORMATA WAVELET DISCRETA (DWT)
            wavelet = 'db4'  # Wavelet Daubechies 4
            coeffs = pywt.wavedec(signal, wavelet, level=4)
            
            # Ricostruzione dei dettagli a ogni livello
            cA4, cD4, cD3, cD2, cD1 = coeffs  # Coefficienti delle approssimazioni e dettagli
            
            # Grafico del segnale originale
            plt.figure(figsize=(12, 8))
            
            plt.subplot(3, 1, 1)
            plt.plot(t, signal, label="Segnale Originale")
            plt.title("Segnale nel dominio del tempo")
            plt.xlabel("Tempo (s)")
            plt.ylabel("Ampiezza")
            plt.legend()
            
            # Grafico della FFT
            plt.subplot(3, 1, 2)
            plt.plot(positive_frequencies, fft_magnitudes, label="FFT", color='r')
            plt.title("Trasformata di Fourier (FFT)")
            plt.xlabel("Frequenza (Hz)")
            plt.ylabel("Ampiezza")
            plt.legend()
            
            # Grafico dei coefficienti Wavelet
            plt.subplot(3, 1, 3)
            plt.plot(cD1, label="Dettaglio Livello 1")
            plt.plot(cD2, label="Dettaglio Livello 2")
            plt.plot(cD3, label="Dettaglio Livello 3")
            plt.plot(cD4, label="Dettaglio Livello 4")
            plt.title("Coefficienti della Trasformata Wavelet (DWT)")
            plt.xlabel("Campioni")
            plt.ylabel("Ampiezza")
            plt.legend()
            
            plt.tight_layout()
            plt.show()
            n +=1
        if n == 1:
            break
    client.communicate(10)
client.deregister_event_request(buffer_handle, request_id)
client.disconnect()
del client, buffer_handle, request_id

In [ ]:
k_B = 1.38e-23  # Boltzmann constant
Teff = 3
AmpThermalNoise = 1
SampligTime_s  = 1.4285714285714286e-09
t_max = SampligTime_s * 1024
t = np.arange(0, t_max, SampligTime_s)
eta_thermal = AmpThermalNoise * np.random.normal(0, np.sqrt(k_B * Teff), len(t))

In [ ]:
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.plot(t, eta_thermal)
plt.xlabel('Time (s)')
plt.ylabel('Displacement x(t)')
plt.title('Displacement')
plt.grid(True)

In [ ]:
n_steps=1024
dt = SampligTime_s
xi = np.random.normal(0, 1, n_steps) / np.sqrt(dt)
D = 0.5           # Intensità del rumore
tau_c = 0.1       # Tempo di correlazione
F_noise = np.zeros(n_steps)
for i in range(1, n_steps):
    F_noise[i] = F_noise[i-1] * np.exp(-dt/tau_c) + np.sqrt(D * dt) * xi[i]

In [ ]:
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.plot(t, F_noise)
plt.xlabel('Time (s)')
plt.ylabel('Displacement x(t)')
plt.title('Displacement')
plt.grid(True)

In [ ]:
json_data = '''
{
    "nome": "Giovanni",
    "eta": 59,
    "citta": "Roma"
}
'''
class Persona:
    def __init__(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)


In [ ]:
import midas.client

"""
A simple example program that connects to a midas experiment,
reads an ODB value, then sets an ODB value.

Expected output is:

```
The experiment is currently stopped
The new value of /pyexample/eg_float is 5.670000
```
"""

if __name__ == "__main__":
    client = midas.client.MidasClient("pytest")

    # Read a value from the ODB. The return value is a normal python
    # type (an int in this case, but could also be a float, string, bool,
    # list or dict).
    state = client.odb_get("/Runinfo/State")

    if state == midas.STATE_RUNNING:
        print("The experiment is currently running")
    elif state == midas.STATE_PAUSED:
        print("The experiment is currently paused")
    elif state == midas.STATE_STOPPED:
        print("The experiment is currently stopped")
    else:
        print("The experiment is in an unexpected run state")


    # Update or create a directory in the ODB by passing a dict to `odb_set`
    client.odb_set("/pyexample", {"an_int": 1, "eg_float": 4.56})

    # Update a single value in the ODB
    client.odb_set("/pyexample/eg_float", 5.67)

    # Read the value back
    readback = client.odb_get("/pyexample/eg_float")

    print("The new value of /pyexample/eg_float is %f" % readback)

    # Delete the temporary directory we created
    client.disconnect()
    del client
    #client.odb_delete("/pyexample")

In [ ]:
client.disconnect()